In [68]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from qdrant_client.http.models import SearchParams
from getpass import getpass
from datetime import datetime, timezone
from typing import Any
from pathlib import Path
import pandas as pd
import os
import json

In [72]:
COLLECTION_NAME = "20260424_big_pdfs_removed"
DENSE_MODEL_NAME = "BAAI/bge-m3"
SPARSE_MODEL_NAME = "Qdrant/bm25"
K = 10

if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)
avg_len_fr = 254.62018086625417
sparse_fr = FastEmbedSparse(model_name="Qdrant/bm25", avg_len= avg_len_fr, language="french")

qdrant = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    embedding=embeddings,
    sparse_embedding=sparse_fr,
    collection_name=COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse_fr"
)

In [73]:
def _to_python(obj):
    if obj is None:
        return None

    if isinstance(obj, dict):
        return {str(k): _to_python(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [_to_python(x) for x in obj]

    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass

    if hasattr(obj, "tolist"):
        try:
            return obj.tolist()
        except Exception:
            pass

    return obj

def _normalize_sparse_embedding(sparse_emb):
    if sparse_emb is None:
        return {"indices": [], "values": []}

    if isinstance(sparse_emb, dict):
        indices = sparse_emb.get("indices", [])
        values = sparse_emb.get("values", [])
        return {
            "indices": _to_python(indices),
            "values": _to_python(values),
        }

    indices = getattr(sparse_emb, "indices", [])
    values = getattr(sparse_emb, "values", [])
    return {
        "indices": _to_python(indices),
        "values": _to_python(values),
    }

In [ ]:
output_dir = Path("new_dev_dataset_results")
output_dir.mkdir(parents=True, exist_ok=True)

dataset = dataset = pd.read_csv('../test/new_tb_panchaud_jeu_test - saskya_processing.csv', header=0)

manifest_path = output_dir / "manifest.json"
results_path = output_dir / "dev_queries_results.jsonl"

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "collection_name": COLLECTION_NAME,
    "dense_model_name": DENSE_MODEL_NAME,
    "sparse_model_name": SPARSE_MODEL_NAME,
    "params_retrieval": {"k": K, "exact": True}
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

n_exported = 0

with (open(results_path, "w", encoding="utf-8") as jsonl_file):
    for row_idx, row in dataset.iterrows():
        if (
            row.get("Dataset") == "dev"
            and pd.notna(row.get("Id document"))
            and pd.notna(row.get("Question"))
        ):
            query = str(row["Question"]).strip()

            dense_embedding = _to_python(embeddings.embed_query(query))
            sparse_embedding = _normalize_sparse_embedding(sparse_fr.embed_query(query))

            hits = qdrant.similarity_search_with_score(query, k=K, search_params=SearchParams(exact=True))

            top_chunks = []
            for rank, (chunk, score) in enumerate(hits, start=1):
                top_chunks.append(
                    {
                        "rank": rank,
                        "score": _to_python(score),
                        "chunk_text": getattr(chunk, "page_content", ""),
                        "chunk_metadata": _to_python(getattr(chunk, "metadata", {})),
                    }
                )

            result_record = {
                "question": query,
                "question_dense_embedding": dense_embedding,
                "question_sparse_embedding": sparse_embedding,
                "top_chunks": top_chunks,
            }
            jsonl_file.write(
                json.dumps(result_record, ensure_ascii=False) + "\n"
            )
            n_exported += 1

print(f"Exported {n_exported} queries with corresponding results in {manifest_path} and {results_path}")